In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

import os
import pandas as pd
import pgserver

from sqlalchemy import create_engine, text

pg = pgserver.get_server("/tmp/hw5_pg", cleanup_mode="stop")

engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())


Note: you may need to restart the kernel to use updated packages.
PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


In [2]:
%load_ext sql

%config SqlMagic.autopandas = True

%sql $engine.url


In [4]:
import kagglehub

dataset_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Dataset path:", dataset_path)
print("Files:", os.listdir(dataset_path))


Dataset path: /Users/macbook/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2
Files: ['olist_sellers_dataset.csv', 'product_category_name_translation.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv']


In [5]:
import os
import kagglehub

# Завантажуємо датасет Olist
dataset_dir = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Dataset directory:", dataset_dir)
print("\nAvailable CSV files:")

for file in sorted(os.listdir(dataset_dir)):
    if file.endswith(".csv"):
        print(file)


Dataset directory: /Users/macbook/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2

Available CSV files:
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [7]:
files = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)

    df = pd.read_csv(path)

    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10_000)

    print(table_name, df.shape)


olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


In [9]:

%%sql

DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders CASCADE;
DROP TABLE IF EXISTS olist_customers CASCADE;
DROP TABLE IF EXISTS olist_products CASCADE;
DROP TABLE IF EXISTS olist_sellers CASCADE;

CREATE TABLE olist_customers (
    customer_id TEXT PRIMARY KEY,
    customer_unique_id TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city TEXT,
    customer_state CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL
        REFERENCES olist_customers(customer_id),
    order_status TEXT NOT NULL,
    order_purchase_timestamp TIMESTAMP NOT NULL,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id TEXT PRIMARY KEY,
    product_category_name TEXT,
    product_name_length INTEGER
        CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER
        CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty INTEGER
        CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g INTEGER
        CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm INTEGER
        CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm INTEGER
        CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm INTEGER
        CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city TEXT,
    seller_state CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id TEXT NOT NULL
        REFERENCES olist_orders(order_id),
    order_item_id INTEGER NOT NULL,
    product_id TEXT
        REFERENCES olist_products(product_id),
    seller_id TEXT
        REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price NUMERIC(12, 2)
        CHECK (price IS NULL OR price >= 0),
    freight_value NUMERIC(12, 2)
        CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.


""


In [10]:

%%sql

INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
FROM olist_customers_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght,
    product_description_lenght,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state
FROM olist_sellers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price,
    freight_value
FROM olist_order_items_raw;


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
99441 rows affected.
32951 rows affected.
3095 rows affected.
99441 rows affected.
112650 rows affected.


""
